# GAVE2 V8: R2-V2 Direct and Gated Graph Submission

This notebook produces a reproducible direct submission from the released GAVE-winning R2-V2 `av` and `bv` weights. It then evaluates an optional crossing-aware graph projection and creates a second submission only when the graph passes Dice, sensitivity, and observed-score gates.

Both released checkpoints are CFP-only models. S006 intentionally uses the same CFP-derived probabilities for Task 1 and Task 2 as a clean transfer test; it does not use GAVE2 FFA inputs.

The pipeline never crops the fundus canvas. R2-V2 uses its released internal width of 1408 for checkpoint compatibility and restores every probability map to the native 1536 x 1024 submission size.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

DRIVE_BASE = Path("/content/drive/MyDrive/MICCAI2026")
ARCHIVE_PATH = DRIVE_BASE / "miccai_v8.zip"
EXPECTED_ARCHIVE_SHA256 = "4003158544825cdcfe2b2ac8f7e2a2240055fe7ed0e4857a812fc237760e521e"
WORK_ROOT = Path("/content/MICCAI2026")
RUN_DIR = DRIVE_BASE / "runs/gave2_r2v2_v8"
WEIGHTS_DIR = DRIVE_BASE / "cache/r2v2_v1"
SOURCE_DIR = Path("/content/R2-V2")
SUBMISSION_ROOT = DRIVE_BASE / "submissions/gave2_v8"

TEAM_ID = "梯度不下降队"
DIRECT_SUBMISSION_ID = "GAVE2-S006"
GRAPH_SUBMISSION_ID = "GAVE2-S007"
RESIZE_WIDTH = 1408
RUN_GRAPH_SEARCH = True
AUTO_DISCONNECT = True

RUN_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_ROOT.mkdir(parents=True, exist_ok=True)
assert ARCHIVE_PATH.exists(), ARCHIVE_PATH


## Extract clean source and data

The archive contains the local GAVE2 data, V8 source, the shared submission validator, and focused V8 tests. Previous run directories and checkpoints are not imported.


In [ ]:
def sha256_file(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()

archive_sha256 = sha256_file(ARCHIVE_PATH)
assert archive_sha256 == EXPECTED_ARCHIVE_SHA256, (archive_sha256, EXPECTED_ARCHIVE_SHA256)

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    names = set(archive.namelist())
    required = {
        "GAVE2_preliminary/training/images/g_001.png",
        "GAVE2_preliminary/validation/images/g_051.png",
        "experiments/gave2_v8/predict_r2v2.py",
        "experiments/gave2_v8/assets/proven_task3/g_051.txt",
        "experiments/gave2_ensemble/submission_v6.py",
    }
    assert required.issubset(names), sorted(required - names)
    if WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)
    WORK_ROOT.mkdir(parents=True)
    archive.extractall(WORK_ROOT)

DATA_ROOT = WORK_ROOT / "GAVE2_preliminary"
TASK3_SOURCE = WORK_ROOT / "experiments/gave2_v8/assets/proven_task3"
assert len(list((DATA_ROOT / "training/images").glob("*.png"))) == 50
assert len(list((DATA_ROOT / "validation/images").glob("*.png"))) == 50
assert len(list(TASK3_SOURCE.glob("*.txt"))) == 50
print({"archive_sha256": archive_sha256, "work_root": str(WORK_ROOT)})


## Runtime and tests

BF16 is used when supported; otherwise CUDA FP16 is selected. The implementation is not tied to a specific GPU model.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "experiments/gave2_v8/requirements.txt"],
    cwd=WORK_ROOT,
    check=True,
)

import torch
assert torch.cuda.is_available(), "R2-V2 inference is technically CPU-compatible but this notebook requires a CUDA runtime"
AMP = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
gpu = torch.cuda.get_device_properties(0)
print({
    "torch": torch.__version__,
    "gpu": gpu.name,
    "vram_gib": round(gpu.total_memory / 1024**3, 2),
    "amp": AMP,
})

subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests/gave2_v8", "-v"],
    cwd=WORK_ROOT,
    check=True,
)


In [ ]:
def run_module(module: str, *arguments) -> None:
    command = [sys.executable, "-m", module, *map(str, arguments)]
    print("RUN:", " ".join(command))
    subprocess.run(command, cwd=WORK_ROOT, check=True)

VALIDATION_AV = RUN_DIR / "predictions/validation/av"
VALIDATION_BV = RUN_DIR / "predictions/validation/bv"
VALIDATION_DIRECT = RUN_DIR / "predictions/validation/direct"
TRAINING_AV = RUN_DIR / "predictions/training/av"
TRAINING_BV = RUN_DIR / "predictions/training/bv"
TRAINING_DIRECT = RUN_DIR / "predictions/training/direct"
GRAPH_SELECTION = RUN_DIR / "graph_selection.json"
VALIDATION_GRAPH = RUN_DIR / "predictions/validation/graph"


## Acquire pinned R2-V2 source and weights

The two checkpoints total about 500 MB. Downloads resume from `.part` files and every asset is checked against its release SHA256.


In [ ]:
run_module(
    "experiments.gave2_v8.assets",
    "--source-dir", SOURCE_DIR,
    "--weights-dir", WEIGHTS_DIR,
)


## Validation inference and direct fusion

The corrected eight-view TTA transforms the rectangular ROI mask together with the image. Inference is case-resumable on Drive.


In [ ]:
for model_type, output_store in (("av", VALIDATION_AV), ("bv", VALIDATION_BV)):
    run_module(
        "experiments.gave2_v8.predict_r2v2",
        "--data-root", DATA_ROOT,
        "--source-dir", SOURCE_DIR,
        "--weights-dir", WEIGHTS_DIR,
        "--output-store", output_store,
        "--model-type", model_type,
        "--split", "validation",
        "--amp", AMP,
        "--resize-width", RESIZE_WIDTH,
        "--tta",
    )

run_module(
    "experiments.gave2_v8.fuse",
    "--av-store", VALIDATION_AV,
    "--bv-store", VALIDATION_BV,
    "--output-store", VALIDATION_DIRECT,
    "--split", "validation",
    "--bv-class-weight", 0.0,
    "--vessel-mode", "max",
)


## Build S006 direct candidate

This is the first candidate to submit. It changes Task 1 and Task 2 to R2-V2 while keeping the leaderboard-proven Task 3 output fixed.


In [ ]:
direct_zip = SUBMISSION_ROOT / f"{DIRECT_SUBMISSION_ID}__v8-r2v2-direct__{TEAM_ID}.zip"
if direct_zip.exists():
    print("Direct candidate already certified:", direct_zip)
else:
    run_module(
        "experiments.gave2_v8.submission",
        "--data-root", DATA_ROOT,
        "--task1-store", VALIDATION_DIRECT,
        "--task2-store", VALIDATION_DIRECT,
        "--namespace", "r2v2_direct",
        "--task3-source", TASK3_SOURCE,
        "--output-root", SUBMISSION_ROOT,
        "--team-id", TEAM_ID,
        "--submission-id", DIRECT_SUBMISSION_ID,
        "--version", "v8-r2v2-direct",
    )
assert direct_zip.exists(), direct_zip
print("DIRECT SUBMISSION:", direct_zip)


## Optional graph gate

The following cells run R2-V2 on the 50 labeled cases, score the direct output, and search conservative graph parameters. Candidate arrays are temporary. The selected graph must improve the empirically observed score by at least 0.10 while limiting Dice loss to 0.03 and sensitivity loss to 0.02.


In [ ]:
if RUN_GRAPH_SEARCH:
    for model_type, output_store in (("av", TRAINING_AV), ("bv", TRAINING_BV)):
        run_module(
            "experiments.gave2_v8.predict_r2v2",
            "--data-root", DATA_ROOT,
            "--source-dir", SOURCE_DIR,
            "--weights-dir", WEIGHTS_DIR,
            "--output-store", output_store,
            "--model-type", model_type,
            "--split", "training",
            "--amp", AMP,
            "--resize-width", RESIZE_WIDTH,
            "--tta",
        )
    run_module(
        "experiments.gave2_v8.fuse",
        "--av-store", TRAINING_AV,
        "--bv-store", TRAINING_BV,
        "--output-store", TRAINING_DIRECT,
        "--split", "training",
        "--bv-class-weight", 0.0,
        "--vessel-mode", "max",
    )


In [ ]:
if RUN_GRAPH_SEARCH:
    graph_work = Path("/content/gave2_v8_graph_search")
    graph_work.mkdir(parents=True, exist_ok=True)
    if not GRAPH_SELECTION.exists():
        run_module(
            "experiments.gave2_v8.graph", "search",
            "--data-root", DATA_ROOT,
            "--input-store", TRAINING_DIRECT,
            "--work-dir", graph_work,
            "--output", GRAPH_SELECTION,
            "--paths-per-case", 60,
            "--seed", 77,
            "--minimum-gain", 0.10,
            "--maximum-dice-drop", 0.03,
            "--maximum-sensitivity-drop", 0.02,
        )
    selection = json.loads(GRAPH_SELECTION.read_text())
    summary = {
        "accepted": selection["accepted"],
        "gain": selection["gain"],
        "dice_drop": selection["dice_drop"],
        "sensitivity_drop": selection["sensitivity_drop"],
        "direct_score": selection["direct_report"]["score_observed"],
        "selected_score": selection["selected"]["report"]["score_observed"],
        "direct_cor": selection["direct_report"]["mean"]["cor"],
        "selected_cor": selection["selected"]["report"]["mean"]["cor"],
        "direct_inf": selection["direct_report"]["mean"]["inf"],
        "selected_inf": selection["selected"]["report"]["mean"]["inf"],
    }
    print(json.dumps(summary, indent=2))


## Build S007 only if the graph passes

A rejected graph produces no second ZIP. The direct candidate remains the submission recommendation.


In [ ]:
graph_zip = SUBMISSION_ROOT / f"{GRAPH_SUBMISSION_ID}__v8-r2v2-graph__{TEAM_ID}.zip"
if RUN_GRAPH_SEARCH and selection["accepted"]:
    run_module(
        "experiments.gave2_v8.graph", "apply",
        "--input-store", VALIDATION_DIRECT,
        "--output-store", VALIDATION_GRAPH,
        "--selection", GRAPH_SELECTION,
        "--split", "validation",
    )
    if graph_zip.exists():
        print("Graph candidate already certified:", graph_zip)
    else:
        run_module(
            "experiments.gave2_v8.submission",
            "--data-root", DATA_ROOT,
            "--task1-store", VALIDATION_GRAPH,
            "--task2-store", VALIDATION_GRAPH,
            "--namespace", "r2v2_graph",
            "--task3-source", TASK3_SOURCE,
            "--output-root", SUBMISSION_ROOT,
            "--team-id", TEAM_ID,
            "--submission-id", GRAPH_SUBMISSION_ID,
            "--version", "v8-r2v2-graph",
        )
    assert graph_zip.exists(), graph_zip
    print("GRAPH SUBMISSION:", graph_zip)
else:
    print("Graph gate rejected. Submit only:", direct_zip)


## Final record

Record S006 on the leaderboard before submitting any accepted graph candidate. That result measures transfer from the public GAVE-winning weights without confounding Task 3 or graph postprocessing.


In [ ]:
final_record = {
    "direct_zip": str(direct_zip),
    "graph_zip": str(graph_zip) if graph_zip.exists() else None,
    "graph_selection": str(GRAPH_SELECTION) if GRAPH_SELECTION.exists() else None,
    "amp": AMP,
    "resize_width": RESIZE_WIDTH,
    "task3_source": "leaderboard-proven V6 refined",
}
print(json.dumps(final_record, indent=2, ensure_ascii=False))

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()
